# Tạo DSS System nhanh
Notebook này tạo file airline_dss_system.pkl từ các models có sẵn

In [1]:
import pickle
import pandas as pd
import numpy as np
from typing import Dict
import warnings
warnings.filterwarnings('ignore')

print("Loading models and weights...")

Loading models and weights...


In [2]:
# Load existing models
models = {}

# Try loading Business model
try:
    with open('business_xgboost_optimized.pkl', 'rb') as f:
        models['Business'] = pickle.load(f)
    print("✓ Business model loaded")
except FileNotFoundError:
    print("⚠ Business model not found - will use None")
    models['Business'] = None

# Try loading Economy model
try:
    with open('economy_xgboost_optimized.pkl', 'rb') as f:
        models['Economy'] = pickle.load(f)
    print("✓ Economy model loaded")
except FileNotFoundError:
    print("⚠ Economy model not found - will use None")
    models['Economy'] = None

# Try loading Eco Plus model
try:
    with open('ecoplus_xgboost_optimized.pkl', 'rb') as f:
        models['Eco Plus'] = pickle.load(f)
    print("✓ Eco Plus model loaded")
except FileNotFoundError:
    print("⚠ Eco Plus model not found - will use None")
    models['Eco Plus'] = None

✓ Business model loaded
✓ Economy model loaded
✓ Eco Plus model loaded


In [3]:
# Load AHP weights
try:
    with open('ahp_weights_all_classes.pkl', 'rb') as f:
        ahp_weights = pickle.load(f)
    print("✓ AHP weights loaded")
except FileNotFoundError:
    print("❌ AHP weights not found - please run ahp_analysis.ipynb first")
    ahp_weights = None

✓ AHP weights loaded


In [4]:
# Feature configurations
FEATURE_CONFIG = {
    'Business': [
        'Inflight wifi service',
        'Departure/Arrival time convenient',
        'Ease of Online booking',
        'Gate location',
        'Baggage handling',
        'On-board service'
    ],
    'Economy': [
        'Food and drink',
        'Inflight entertainment',
        'Ease of Online booking',
        'Online boarding',
        'Seat comfort',
        'Cleanliness'
    ],
    'Eco Plus': [
        'Seat comfort',
        'Leg room service',
        'Cleanliness',
        'Inflight service',
        'Food and drink',
        'Inflight entertainment'
    ]
}

print("Feature configurations defined")

Feature configurations defined


In [5]:
# Define AirlineDSS class (simplified version)
class AirlineDSS:
    def __init__(self, models_dict, ahp_weights_dict, feature_config):
        self.models = models_dict
        self.ahp_weights = ahp_weights_dict
        self.feature_config = feature_config
        
    def predict_satisfaction(self, passenger_data: pd.DataFrame, ticket_class: str) -> Dict:
        if ticket_class not in self.models:
            raise ValueError(f"Invalid ticket class: {ticket_class}")
        
        model = self.models[ticket_class]
        if model is None:
            raise ValueError(f"Model for {ticket_class} not loaded")
        
        features = self.feature_config[ticket_class]
        X = passenger_data[features]
        
        prediction = model.predict(X)[0]
        probabilities = model.predict_proba(X)[0]
        
        return {
            'prediction': 'Satisfied' if prediction == 1 else 'Dissatisfied',
            'confidence': max(probabilities) * 100,
            'prob_dissatisfied': probabilities[0] * 100,
            'prob_satisfied': probabilities[1] * 100
        }
    
    def calculate_feature_impact(self, passenger_data: pd.DataFrame, ticket_class: str) -> pd.DataFrame:
        model = self.models[ticket_class]
        features = self.feature_config[ticket_class]
        
        xgb_importance = model.feature_importances_
        
        ahp_dict = self.ahp_weights[ticket_class]['weights']
        ahp_values = np.array([ahp_dict[f] for f in features])
        
        feature_values = passenger_data[features].values[0]
        
        combined_weights = (xgb_importance * 0.5 + ahp_values * 0.5)
        impact_scores = combined_weights * feature_values
        
        impact_percentage = (impact_scores / impact_scores.sum()) * 100
        
        impact_df = pd.DataFrame({
            'Feature': features,
            'Current_Value': feature_values,
            'XGBoost_Importance': xgb_importance,
            'AHP_Weight': ahp_values,
            'Combined_Weight': combined_weights,
            'Impact_Score': impact_scores,
            'Impact_%': impact_percentage
        })
        
        impact_df = impact_df.sort_values('Impact_Score', ascending=False).reset_index(drop=True)
        
        return impact_df
    
    def generate_recommendations(self, passenger_data: pd.DataFrame, ticket_class: str) -> Dict:
        prediction_result = self.predict_satisfaction(passenger_data, ticket_class)
        impact_df = self.calculate_feature_impact(passenger_data, ticket_class)
        
        weak_features = impact_df[
            (impact_df['Current_Value'] <= 2) & 
            (impact_df['Combined_Weight'] > impact_df['Combined_Weight'].median())
        ]
        
        strong_features = impact_df[
            (impact_df['Current_Value'] >= 4) & 
            (impact_df['Combined_Weight'] > impact_df['Combined_Weight'].median())
        ]
        
        priority_actions = []
        for idx, row in weak_features.iterrows():
            priority_actions.append({
                'feature': row['Feature'],
                'current_rating': row['Current_Value'],
                'importance': row['Combined_Weight'],
                'impact': row['Impact_%'],
                'urgency': 'HIGH' if row['Current_Value'] <= 1 else 'MEDIUM',
                'action': f"Improve '{row['Feature']}' (currently {row['Current_Value']}/5)"
            })
        
        risk_score = (prediction_result['prob_dissatisfied'] * 0.7 + 
                     len(weak_features) / len(impact_df) * 100 * 0.3)
        
        return {
            'ticket_class': ticket_class,
            'prediction': prediction_result,
            'impact_analysis': impact_df,
            'weak_features': weak_features,
            'strong_features': strong_features,
            'priority_actions': priority_actions,
            'risk_score': risk_score,
            'risk_level': 'HIGH' if risk_score >= 70 else 'MEDIUM' if risk_score >= 40 else 'LOW'
        }

print("✓ AirlineDSS class defined")

✓ AirlineDSS class defined


In [6]:
# Create DSS instance
if ahp_weights:
    dss = AirlineDSS(
        models_dict=models,
        ahp_weights_dict=ahp_weights,
        feature_config=FEATURE_CONFIG
    )
    print("✓ DSS initialized successfully")
    
    # Show status
    print("\nModels available:")
    for class_name, model in models.items():
        status = "✓ Loaded" if model is not None else "✗ Not available"
        print(f"  {class_name}: {status}")
else:
    print("❌ Cannot create DSS - AHP weights missing")
    dss = None

✓ DSS initialized successfully

Models available:
  Business: ✓ Loaded
  Economy: ✓ Loaded
  Eco Plus: ✓ Loaded


In [7]:
# Save DSS system
if dss:
    with open('airline_dss_system.pkl', 'wb') as f:
        pickle.dump(dss, f)
    
    print("\n" + "="*60)
    print("✅ SUCCESS: DSS system saved to airline_dss_system.pkl")
    print("="*60)
    print("\nYou can now:")
    print("1. Reload the Streamlit web app (press R in browser)")
    print("2. The error should be gone")
    print("3. Start using the Customer Form")
    print("\nNote: Only classes with loaded models will work in the web app.")
else:
    print("\n❌ FAILED: Could not create DSS system")
    print("Please ensure ahp_weights_all_classes.pkl exists")


✅ SUCCESS: DSS system saved to airline_dss_system.pkl

You can now:
1. Reload the Streamlit web app (press R in browser)
2. The error should be gone
3. Start using the Customer Form

Note: Only classes with loaded models will work in the web app.
